In [21]:
from data_extraction.utils.rate_limit_handling import request_with_backoff, RateLimiter, request_m4a_with_backoff
import requests
import os
from loguru import logger
from data_extraction.db_operations.get_features import execute

results = execute("select id, name, name_en from artists")
results = results[10:]

In [22]:
from data_extraction.utils.rate_limit_handling import RateLimiter, request_with_backoff
import os
lastfm_limiter = RateLimiter(calls_per_second=2)
def get_results(name: str):
    LASTFM_API_ENDPOINT = "https://ws.audioscrobbler.com/2.0/"
    LAST_FM_API_KEY = os.getenv("LAST_FM_API_KEY")
    endpoint = "artist.search"
    headers = {"User-Agent": "ArabMusicMap/1.0 (yussef0212@gmail.com)"}
    params = {
        "method": endpoint,
        "artist": name,
        "api_key": LAST_FM_API_KEY,
        "format": "json",
        "limit": 5
    }

    lastfm_limiter.wait()
    try:
        response = request_with_backoff(LASTFM_API_ENDPOINT, params=params, headers=headers)
        artist_list = response['results']['artistmatches']['artist']
        data = [(artist['name'], artist['listeners']) for artist in artist_list]
    except Exception as e:
        print(e)
        data = []
    return data

In [23]:
from data_extraction.db_operations.get_features import getcon
con = getcon()
def save_listeners(id, listeners):
    listeners = listeners if listeners else 1
    con.execute("update artists set lastfm_listeners = ? where id = ?", [listeners, id])

In [24]:
import os
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_groq import ChatGroq
from langchain.messages import SystemMessage, HumanMessage
from pydantic import RootModel, BaseModel
from typing import Any
load_dotenv()
class Listeners(BaseModel):
    listeners: int
    reasoning: str
def create_messages2(artists: dict, artist_names) -> list:
    human_message = "Arabic Name: {name}, English Name: {name_en}\nInput: {stuff}"
    template = human_message.format(stuff=str(artists), name=artist_names[0], name_en=artist_names[1])
    messages = [
        SystemMessage(content=
        "Your task is to choose the largest relevant listener count for an artist"
        "You will receive a list of dicts of artists and their listeners count"
        "you will return the largest listener count you can find"
        "but be careful, since since not every artist shown is the correct artist"
        "you will be provided with the english and arabic name of the artist and their existing listener count"
        "the listener count you get must be of an artist that has the same of the names you are provided"
        "it doesnt need to be an exact text match but just has to be the same artist, if the english and arabic name you are given are one word, then the name you choose has to be an exact text match to either of the english or arabic"
        "RETURN THE NUMBER ONLY NOT THE NAME, and reasoning"
        ),
        HumanMessage(content=template)
    ]
    return messages

    
def choose_listeners2(artists: dict, artist_names) -> int:
    model = ChatMistralAI(
        api_key=os.getenv("MISTRAL_API_KEY"),
        model='ministral-8b-2512',
        max_retries=5
    )
    # model = ChatGroq(
    #     api_key=os.getenv("GROQ_API_KEY"),
    #     model='ministral-3b-2512',
    # )
    model = model.with_structured_output(Listeners)
    messages = create_messages2(artists, artist_names)

    response = model.invoke(messages)
    return response

In [25]:
from tqdm import tqdm
from data_extraction.utils.rate_limit_handling import request_with_backoff
from data_extraction.utils.rate_limit_handling import RateLimiter
from data_extraction.db_operations.get_features import getcon
# from translate_names.main import choose_listeners

# agent_inputs = []
temp_results = [
    (762, "Fayçal Azizi", "Fayçal Azizi"),
    (909, "Issam Houshan", "Issam Houshan"),
    (1292, "Riad Awwad", "Riad Awwad"),
    (1351, "Imane Homsy", "Imane Homsy"),
    (1789, "Trio Abozekrys", "Trio Abozekrys")
    
    ]
# responses += temp_results
llm_limiter = RateLimiter(calls_per_second=2)
responses = []
fails = []
for id, name, name_en in tqdm(results):
    if name_en == name:
        agent_input = get_results(name)
    else:
        agent_input = get_results(name)
        agent_input += get_results(name_en)
    if agent_input:
        llm_limiter.wait()
        try:
            response = choose_listeners2(agent_input, (name, name_en))
            responses.append((id, response))
        except Exception as e:
            print(e)
            fails.append(((id, name, name_en), e, agent_input))

for id, name, name_en in temp_results:
    agent_input = get_results(name)
    llm_limiter.wait()
    response = choose_listeners2(agent_input, (name, name_en))
    responses.append((id, response))
import json
json_responses = []
for item in responses:
    json_responses.append((item[0], item[1].listeners))
with open("revised_listeners.json", "w") as f:
    json.dump(json_responses, f)

100%|██████████| 1081/1081 [38:11<00:00,  2.12s/it] 


In [39]:
import json
json_responses = []
for item in safe_responses:
    json_responses.append((item[0], item[1].listeners))
with open("listeners.json", "w") as f:
    json.dump(json_responses, f)
temp_results = [
    (762, "Fayçal Azizi", "Fayçal Azizi"),
    (909, "Issam Houshan", "Issam Houshan"),
    (1292, "Riad Awwad", "Riad Awwad"),
    (1351, "Imane Homsy", "Imane Homsy"),
    (1789, "Trio Abozekrys", "Trio Abozekrys")
    
    ]
for id, name, name_en in temp_results:
    agent_input = get_results(name)
    llm_limiter.wait()
    response = choose_listeners2(agent_input, (name, name_en))
    safe_responses.append((id, response))
import json
json_responses = []
for item in safe_responses:
    json_responses.append((item[0], item[1].listeners))
with open("listeners.json", "w") as f:
    json.dump(json_responses, f)

In [33]:
response = choose_listeners2(fails[1][2], fails[2][0])
response

Listeners(listeners=5956)

In [ ]:
import json
json_responses = []
for item in responses:
    json_responses.append((item[0], item[1].listeners))
with open("listeners.json", "w") as f:
    json.dump(json_responses, f)

In [27]:
for id, value in responses:
    # print(id, value.listeners)
    # break
    save_listeners(id, value.listeners)

In [19]:
# gaaa = []
# for _, names in agent_inputs:
#     name, name_en = names
#     gaaa.append(name)
# gaa = []
# for id, name, name_en in results:
#     gaa.append((id, name))

# not_in_gaaa = [(id, name) for id, name in gaa if name not in gaaa]
# # not_in_gaaa now contains (id, name) tuples from results whose name is not in gaaa
not_in_gaaa

[(733, 'ماريان واغدي'),
 (762, 'فيصل عزيزي'),
 (909, 'عصام هوشان'),
 (912, 'مهدي تركي'),
 (1229, 'جماعة موسيقيين'),
 (1292, 'رياض عواد'),
 (1351, 'إيمان حمصي'),
 (1606, 'علي بوشناق'),
 (1789, 'ثلاثي أبو زكري'),
 (2089, 'أنور الجبري')]

In [21]:
for id, name in not_in_gaaa:
    results = execute(f'select id, name, itunes_artist_id from artists where id={id}')
    print(results[0])


(733, 'ماريان واغدي', '1437381840')
(762, 'فيصل عزيزي', '916385907')
(909, 'عصام هوشان', '151032287')
(912, 'مهدي تركي', '1500168696')
(1229, 'جماعة موسيقيين', '1535484822')
(1292, 'رياض عواد', '1715569417')
(1351, 'إيمان حمصي', '207380674')
(1606, 'علي بوشناق', '1840274949')
(1789, 'ثلاثي أبو زكري', '1437387260')
(2089, 'أنور الجبري', '1686770004')


In [26]:
id=2089
execute(
    f"""
delete from artist_similarity_lastfm where artist_id={id};
delete from failed_artists where artist_id={id};
delete from failed_tracks where artist_id={id};
delete from tracks where artist_id={id};
delete from artists where id = {id};
""")

[(1,)]

In [18]:
len(results)

1096

In [37]:
responses = []
from data_extraction.utils.rate_limit_handling import RateLimiter
from organize_artists import create_messages, choose_listeners
llm_limiter = RateLimiter(calls_per_second=0.1)
for item in tqdm(agent_inputs):
    messages = create_messages(item[0], item[1])
    llm_limiter.wait()
    response = choose_listeners(messages)
    responses.append(response)

  0%|          | 0/1 [00:00<?, ?it/s]


TypeError: choose_listeners() missing 1 required positional argument: 'artist_names'

In [5]:
import json
from data_extraction.db_operations.get_features import getcon
with open("translate_names/export.json", 'r') as f:
    data = json.load(f)
# print(data[0])
con = getcon()
for item in data:
    con.execute("update artists set name_en=? where id=?", (item['name_en'], item['id']))


In [7]:
results

[('Samar', '820'),
 ('Kendrick Lamar', '5229270'),
 ('Şamar', '11'),
 ('Samar Jafri', '11241'),
 ('SaMar Tarik', '6380'),
 ('سمر', '54'),
 ('سمر طارق', '20'),
 ('سمر العلي', '7'),
 ('Tamer Nafar, ايتامار تسيچلر, سمر قبطي', '6'),
 ('ملتقى سمر Moltaka Samr', '3')]

In [6]:
results = get_results("Samar") + get_results("سمر")
for item in results:
    print(item)

('Samar', '820')
('Kendrick Lamar', '5229270')
('Şamar', '11')
('Samar Jafri', '11241')
('SaMar Tarik', '6380')
('سمر', '54')
('سمر طارق', '20')
('سمر العلي', '7')
('Tamer Nafar, ايتامار تسيچلر, سمر قبطي', '6')
('ملتقى سمر Moltaka Samr', '3')


In [17]:
response = choose_listeners2(results, ("سمر", "Samar"))
# print(response.reasoning)
response

Listeners(listeners=820, reasoning="The English name provided is 'Samar' (single word), so the closest exact match is 'Samar' with 820 listeners. Other names like 'Şamar', 'Samar Jafri', or 'SaMar Tarik' do not match exactly. The Arabic name 'سمر' (single word) also matches exactly with 54 listeners, but since the English name is 'Samar' (single word), the priority is given to the exact English match.")